# 04 — Phase 4 : générer un plan de jeu avec Claude

Objectif : transformer l'attribution SHAP (phase 3) en plan de jeu rédigé, pour une composition hypothétique (deux équipes de 5 champions, pas un match réel).

Toute la logique de classement (`composition_fact_ranking`) a été **industrialisée** dans `lol_assistant/game_plan.py` — ce notebook l'utilise, il ne la redéfinit pas. C'est exactement pour ça qu'on avait fait ce travail d'industrialisation en phase 2 : la logique réutilisable vit dans le package, les notebooks orchestrent et expliquent.

## Concept 1 — Classer les faits pour les deux équipes

`load_ranking_artifacts` lit les deux CSV produits par le notebook 3 (`global_shap_importance.csv`, `champion_feature_interactions.csv`) — aucun appel au modèle XGBoost ni à SHAP ici, juste une lecture de fichiers, donc très rapide. `composition_fact_ranking` applique la même logique que dans le notebook 3 (le max par champion, pas la somme).

In [1]:
from lol_assistant.game_plan import load_ranking_artifacts, composition_fact_ranking

global_importance, interactions = load_ranking_artifacts()

your_team = ["Jayce", "Sylas", "Yone", "Yunara", "Lulu"]
enemy_team = ["Garen", "Rengar", "Akali", "Hwei", "Rakan"]

your_ranking = composition_fact_ranking(your_team, global_importance, interactions)
enemy_ranking = composition_fact_ranking(enemy_team, global_importance, interactions)

your_ranking.head(6)[["feature", "key_champion", "team_importance"]]

,feature,key_champion,team_importance
0,voidMonsterKill,Jayce,0.664870
1,firstTurretKilledTime,Yone,0.664481
2,maxLevelLeadLaneOpponent,Sylas,0.655693
3,earliestDragonTakedown,Lulu,0.451573
4,teamRiftHeraldKills,Yunara,0.279357
5,visionScoreAdvantageLaneOpponent,Lulu,0.258550


## Concept 2 — Construire le prompt

`build_game_plan_prompt` transforme les deux classements en texte : les faits de **notre** équipe ("à prioriser") et ceux de l'équipe **adverse** ("à surveiller/contrer" — même analyse légitime appliquée à leurs champions, pas un vrai signal de matchup entraîné, comme convenu). On limite volontairement aux 6 faits les plus importants par équipe (`top_n`) — inutile de noyer le LLM avec les 27, les derniers ont une importance quasi nulle de toute façon.

In [2]:
from lol_assistant.game_plan import build_game_plan_prompt

prompt = build_game_plan_prompt(your_team, enemy_team, your_ranking, enemy_ranking)
print(prompt)

Voici une composition League of Legends à analyser.

Notre équipe : Jayce, Sylas, Yone, Yunara, Lulu
Équipe adverse : Garen, Rengar, Akali, Hwei, Rakan

D'après un modèle entraîné sur des dizaines de milliers de vraies parties, voici les faits de jeu (objectifs, avance en lane, etc.) qui comptent le plus pour NOTRE équipe, avec le champion à qui chaque fait profite le plus (élément statistique, pas une garantie) :
- voidMonsterKill (surtout via Jayce)
- firstTurretKilledTime (surtout via Yone)
- maxLevelLeadLaneOpponent (surtout via Sylas)
- earliestDragonTakedown (surtout via Lulu)
- teamRiftHeraldKills (surtout via Yunara)
- visionScoreAdvantageLaneOpponent (surtout via Lulu)

Et voici les mêmes faits de jeu, mais pour l'équipe ADVERSE — donc ce sur quoi elle va probablement s'appuyer, et qu'on devrait chercher à empêcher ou retarder :
- voidMonsterKill (surtout via Garen)
- maxLevelLeadLaneOpponent (surtout via Akali)
- firstTurretKilledTime (surtout via Rakan)
- earliestDragonTaked

## Concept 3 — Appeler Claude

`generate_game_plan` fait tout le pipeline (classement + prompt + appel API) en un appel. On utilise **Claude Sonnet 5** par défaut (`DEFAULT_MODEL` dans `game_plan.py`) — tâche de rédaction assez "gabarisée" (données structurées → texte), pas besoin du modèle le plus capable ; passer `model="claude-opus-5"` pour la meilleure qualité possible si besoin.

**Nécessite `ANTHROPIC_API_KEY` renseignée dans `.env`** (comme pour Riot en phase 1).

In [3]:
from lol_assistant.game_plan import generate_game_plan

game_plan_text, your_ranking, enemy_ranking = generate_game_plan(your_team, enemy_team)
print(game_plan_text)

# Plan de jeu

**Notre priorité : contrôler le rythme avant 15 minutes.**

Sylas doit jouer agressif en solo lane pour prendre un avantage de niveau clair sur Akali — c'est crucial parce qu'Akali est justement le point d'appui de leur plan (elle veut prendre l'avance en lane ET le contrôle de vision). Si Sylas la freine dès les premières vagues, ça coupe l'élan de toute leur équipe.

En parallèle, Jayce doit prioriser les monstres du Néant (Void Grubs) dès qu'ils spawnent — c'est un axe fort pour nous, et ça prive directement Garen de la même ressource côté adverse.

Yone doit chercher à faire tomber la première tour rapidement, quitte à forcer un peu la voie du milieu avec des roams ou en poussant les vagues efficacement. Ça prive aussi Rakan de son objectif favori et ouvre la carte pour nos autres lanes.

Lulu est la clé sur deux fronts : elle doit poser de la vision tôt autour du premier dragon et accompagner l'équipe pour le prendre au plus vite. C'est un double avantage — elle pre

## Concept 4 — Tester la robustesse : entrées invalides

Comment "tester l'assistant" concrètement : essayer d'autres compositions en changeant `your_team`/`enemy_team` plus haut et en ré-exécutant — mais aussi vérifier qu'une **mauvaise entrée** (faute de frappe, mauvais nombre de champions) est détectée plutôt qu'ignorée silencieusement. Testé et corrigé : à l'origine, un nom de champion mal orthographié était simplement ignoré sans avertissement — le plan généré aurait été subtilement faux sans qu'on s'en rende compte. `composition_fact_ranking` valide maintenant ses entrées (voir `lol_assistant/game_plan.py`).

In [4]:
from lol_assistant.game_plan import composition_fact_ranking

try:
    composition_fact_ranking(["Arhi", "Sylas", "Yone", "Yunara", "Lulu"], global_importance, interactions)
except ValueError as e:
    print("Erreur détectée (attendu) :", e)

Erreur détectée (attendu) : Champion(s) non reconnu(s) : 'Arhi' (vouliez-vous dire 'Ahri' ?)


## Concept 5 — Raccourci de saisie : partir d'une partie en cours

Jusqu'ici, `your_team`/`enemy_team` étaient tapés à la main. `lol_assistant/live_game.py` ajoute une alternative : donner un Riot ID (`gameName#tagLine`), aller chercher sa **partie en cours** via Spectator-V5 (routage plateforme — `euw1`, pas `europe`, comme vu en phase 1), et en extraire les deux compositions de 10 champions.

**Ça reste un raccourci de saisie, pas une analyse de partie réelle** : une fois les 10 noms de champions extraits, tout le reste de l'état de la partie (kills, or, timers, et le résultat final) est jeté. Le pipeline en aval (`composition_fact_ranking`, le prompt, l'appel Claude) ne voit que deux listes de noms — exactement comme s'ils avaient été tapés à la main. C'est la décision produit de la phase 1 : l'assistant répond pour une composition hypothétique, jamais pour un match en particulier.

`get_active_game` renvoie `None` si le joueur n'est pas en partie — cas normal (pas une erreur), donc `get_composition_from_active_game` le traduit en `ValueError` explicite plutôt que de planter avec un 404 brut.</cell>


In [5]:
from lol_assistant.riot_client import RiotClient
from lol_assistant.live_game import get_composition_from_active_game

client = RiotClient()

try:
    live_your_team, live_enemy_team = get_composition_from_active_game(client, "Spear Shot", "1111")
    print("Votre équipe :", live_your_team)
    print("Équipe adverse :", live_enemy_team)
except ValueError as e:
    print("Pas de partie en cours (attendu la plupart du temps) :", e)

Pas de partie en cours (attendu la plupart du temps) : Spear Shot#1111 n'est pas en partie actuellement.


In [6]:
# Si une partie en cours a été trouvée ci-dessus, on peut directement l'utiliser comme
# entrée de generate_game_plan — même fonction que pour une composition tapée à la main.
if "live_your_team" in dir():
    from lol_assistant.game_plan import generate_game_plan

    game_plan_text, _, _ = generate_game_plan(live_your_team, live_enemy_team)
    print(game_plan_text)

---

**Checkpoint** : confirme que le plan de jeu généré est cohérent (mentionne bien les faits clés et les champions associés, pas de contresens), et que l'erreur sur `"Arhi"` s'affiche bien.

**Comment tester par toi-même** : change `your_team`/`enemy_team` dans le Concept 1 avec d'autres champions réels (`data/champion_summary.csv` liste les 173 noms valides), ré-exécute les cellules du notebook dans l'ordre. Essaie aussi volontairement une mauvaise entrée (faute de frappe, 4 champions au lieu de 5) pour vérifier que l'erreur reste claire plutôt que silencieuse.